In [1]:
!pip install musicbrainzngs

import pandas as pd
import requests
import musicbrainzngs
import regex as re
from bs4 import BeautifulSoup
import time
import random
from tqdm.notebook import tqdm
import io

from google.colab import userdata


In [2]:
musicbrainzngs.set_useragent("NLP-Music-Project", "0.1", "your.email@example.com")
musicbrainzngs.set_rate_limit(True)
GENIUS_API_KEY = userdata.get("GENIUS_API_KEY")

In [3]:
def fetch_seed_data():
    """
    Aggregates Rolling Stone's 500 Songs, 500 Albums, and 21st Century lists
    into a unified 'Seed DataFrame'.
    """
    print("--- STAGE 1: Fetching Seed Data ---")
    seeds = []

    # 1. RS 500 Albums (2020)
    url_albums = "https://raw.githubusercontent.com/rodriguezcommaj/rs_500/master/albums.csv"
    df_albums = pd.read_csv(url_albums, header=None, on_bad_lines='skip')
    print(df_albums[:10])
    df_albums.columns = ['Rank', 'Artist', 'Album', 'N/A']
    try:
        for _, row in df_albums.iterrows():
            seeds.append({
                'Input_Artist': row['Artist'],
                'Input_Title': row['Album'],
                # # Handle missing genre columns gracefully
                # 'Review_Context': f"Rank {row['Rank']} in RS 500 Albums."
            })
        print(f"Loaded {len(df_albums)} Albums (some bad rows may have been skipped).")
    except Exception as e:
        print(f"Error loading Albums: {e}")
    return pd.DataFrame(seeds)
    # 2. RS 500 Songs
    # url_songs = "https://gist.githubusercontent.com/keune/0de5c7fb669f7b682874/raw/4b174f86a1d82d61d81395b28373b5269f8c6753/2004%2520Rolling%2520Stone%2520Top%2520500%2520Songs.json"
    # try:
    #     response = requests.get(url_songs)
    #     data_songs = response.json()
    #     for item in data_songs:
    #         seeds.append({
    #             'Source_List': 'RS_500_Songs',
    #             'Input_Artist': item['artistTitle'],
    #             'Input_Title': item['songTitle'],
    #             'Input_Type': 'Song',
    #             'Review_Context': f"Rank {item['rank']} in RS 500 Songs. Released: {item['releaseYear']}."
    #         })
    #     print(f"Loaded {len(data_songs)} Songs.")
    # except Exception as e:
    #     print(f"Error loading Songs: {e}")


## Genius API Integration

Replace AZLyrics scraping with Genius API for more reliable lyrics fetching.

In [4]:
import sqlite3
import re
from bs4 import BeautifulSoup

# Genius API Configuration - Modify these values
DATABASE_PATH = "genius_songs.db"

class GeniusAPI:
    def __init__(self, client_access_token):
        """Initialize Genius API client with access token."""
        self.base_url = "https://api.genius.com"
        self.access_token = client_access_token
        self.headers = {"Authorization": f"Bearer {client_access_token}"}

    def search_song(self, artist, song_title):
        """Search for a song by artist and title."""
        query = f"{artist} {song_title}"
        endpoint = f"{self.base_url}/search"
        params = {"q": query}

        response = requests.get(endpoint, headers=self.headers, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        if data.get("response") and data["response"].get("hits"):
            for hit in data["response"]["hits"]:
                result = hit.get("result", {})
                result_title = result.get("title", "").lower()
                result_artist = result.get("primary_artist", {}).get("name", "").lower()

                # Check if it's a close match
                if song_title.lower() in result_title or result_title in song_title.lower():
                    if artist.lower() in result_artist or result_artist in artist.lower():
                        return result

            # If no exact match, return first result
            return data["response"]["hits"][0]["result"]
        return None

    def get_lyrics(self, song_url):
        """Scrape lyrics from the song page URL."""
        response = requests.get(song_url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")

        # Find lyrics containers
        lyric_divs = soup.find_all(attrs={"data-lyrics-container": True})
        if lyric_divs:
            parts = []
            for div in lyric_divs:
                text = div.get_text(separator="\n", strip=True)

                # Skip translation sections
                lines = text.split("\n")
                if lines and re.match(r"\[.*?[Tt]ranslation.*?\]", lines[0].strip()):
                    continue

                # Remove everything before and including the first [] section
                text = re.sub(r"^.*?\[\w+.*?\]\s*", "", text, flags=re.DOTALL)

                if text.strip():
                    parts.append(text)

            return "\n\n".join(parts).strip()

        # Fallback for old layout
        div = soup.find("div", class_=re.compile(r"lyrics", re.I))
        if div:
            text = div.get_text(separator="\n", strip=True)
            text = re.sub(r"^.*?\[\w+.*?\]\s*", "", text, flags=re.DOTALL)
            return text

        return None

# Initialize the Genius API client
genius_client = GeniusAPI(GENIUS_CLIENT_ACCESS_TOKEN)
print("Genius API client initialized")

Genius API client initialized


In [ ]:
def get_lyrics_genius(artist, title):
    """
    Fetch lyrics using Genius API.
    Returns lyrics text or None if not found.
    """
    try:
        # Search for the song
        song_result = genius_client.search_song(artist, title)

        if not song_result:
            print(f"Song not found: {artist} - {title}")
            return None

        # Get the song URL and scrape lyrics
        song_url = song_result.get("url")
        if not song_url:
            return None

        lyrics = genius_client.get_lyrics(song_url)

        if lyrics:
            print(f"✓ Found lyrics for: {artist} - {title}")
        else:
            print(f"✗ No lyrics found for: {artist} - {title}")

        return lyrics

    except Exception as e:
        print(f"Error fetching lyrics for {artist} - {title}: {e}")
        return None

# Test the function
test_lyrics = get_lyrics_genius("Marvin Gaye", "What's Going On")
if test_lyrics:
    print(f"\nFirst 200 characters:\n{test_lyrics[:200]}...")


Error fetching lyrics for Marvin Gaye - What's Going On: 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Marvin+Gaye+What%27s+Going+On


In [ ]:
def expand_and_align_metadata_with_genius(seed_df):
    """
    Takes the seed dataframe, queries MusicBrainz for MBIDs,
    expands 'Albums' into their constituent 'Tracks',
    and fetches lyrics using Genius API.
    """
    print("\n--- STAGE 2: Expanding Metadata & Albums (with Genius API) ---")
    expanded_data = []

    # Processing loop
    for index, row in tqdm(seed_df.iterrows(), total=seed_df.shape[0], desc="Resolving Metadata"):
        artist = row['Input_Artist']
        title = row['Input_Title']

        # Strict query for the release
        query = f'release:{title} AND artist:{artist} AND primarytype:album'
        result = musicbrainzngs.search_releases(query=query, limit=1)

        if result['release-list']:
            release = result['release-list'][0]
            release_id = release['id']

            # Get tracklist
            release_details = musicbrainzngs.get_release_by_id(release_id, includes=['recordings'])

            if 'medium-list' in release_details['release']:
                for medium in release_details['release']['medium-list']:
                    for track in medium['track-list']:
                        track_title = track['recording']['title']

                        # Fetch lyrics using Genius API
                        lyrics = get_lyrics_genius(artist, track_title)

                        data_dict = {
                            'Song_Title': track_title,
                            'Artist': artist,
                            'Album_Name': title,
                            'Lyrics': lyrics
                        }
                        expanded_data.append(data_dict)

                        # Be polite with API rate limiting
                        time.sleep(0.5)

    return pd.DataFrame(expanded_data)


In [ ]:
# 1. Get Seeds
seed_df = fetch_seed_data()
print(f"Total Seed Items: {len(seed_df)}")

--- STAGE 1: Fetching Seed Data ---
    0                          1                                2   3
0   1                Marvin Gaye                   Whats Going On NaN
1   2             The Beach Boys                       Pet Sounds NaN
2   3              Joni Mitchell                             Blue NaN
3   4              Stevie Wonder         Songs in the Key of Life NaN
4   5                The Beatles                       Abbey Road NaN
5   6                    Nirvana                        Nevermind NaN
6   7              Fleetwood Mac                          Rumours NaN
7   8  Prince and the Revolution                      Purple Rain NaN
8   9                  Bob Dylan              Blood on the Tracks NaN
9  10                Lauryn Hill  The Miseducation of Lauryn Hill NaN
Loaded 487 Albums (some bad rows may have been skipped).
Total Seed Items: 487
Loaded 487 Albums (some bad rows may have been skipped).
Total Seed Items: 487


In [ ]:
# 2. Expand to Songs using Genius API (This takes time!)
# Note: This will result in thousands of rows if run fully.
master_df = expand_and_align_metadata_with_genius(seed_df)
print(f"Total Expanded Songs: {len(master_df)}")
print(f"Songs with lyrics: {master_df['Lyrics'].notna().sum()}")
master_df.head()


--- STAGE 2: Expanding Metadata & Albums (with Genius API) ---


Resolving Metadata:   0%|          | 0/487 [00:00<?, ?it/s]

Error fetching lyrics for Marvin Gaye - What’s Going On: 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Marvin+Gaye+What%E2%80%99s+Going+On
Error fetching lyrics for Marvin Gaye - What’s Happening Brother: 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Marvin+Gaye+What%E2%80%99s+Happening+Brother
Error fetching lyrics for Marvin Gaye - What’s Happening Brother: 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Marvin+Gaye+What%E2%80%99s+Happening+Brother
Error fetching lyrics for Marvin Gaye - Flyin’ High (in the Friendly Sky): 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Marvin+Gaye+Flyin%E2%80%99+High+%28in+the+Friendly+Sky%29
Error fetching lyrics for Marvin Gaye - Flyin’ High (in the Friendly Sky): 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Marvin+Gaye+Flyin%E2%80%99+High+%28in+the+Friendly+Sky%29
Error fetching lyrics for Marvin Gaye - Save the Children: 

KeyboardInterrupt: 

In [ ]:
# Optional: Save to SQLite database for persistence
def save_to_sqlite(df, db_path=DATABASE_PATH):
    """Save the dataframe to SQLite database."""
    conn = sqlite3.connect(db_path)

    # Save to database (replaces existing table)
    df.to_sql('songs', conn, if_exists='replace', index=False)

    print(f"Saved {len(df)} songs to {db_path}")
    conn.close()

# Uncomment to save:
# save_to_sqlite(master_df)